# AgriNexus AI — Production Disease Detection Model Training (Google Colab)
This notebook resumes the Three-Phase training of **ConvNeXt-Base (61 classes)** on an NVIDIA T4 GPU.

### Prerequisites:
1. Go to **Runtime → Change runtime type** and set Hardware Accelerator to **T4 GPU**.
2. Upload `agrinexus_training_code.zip` and `agrinexus_dataset.zip` to the Files panel on the left (or via Google Drive).

In [ ]:
# 1. Verify NVIDIA T4 GPU
!nvidia-smi

Fri Sep 11 10:02:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 2. Install Required Production ML Libraries
!pip install -q "albumentations>=1.4.0" "torchmetrics>=1.4.0" "timm>=1.0.3" "scikit-learn>=1.5.0" pyyaml pillow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 57.3 MB/s eta 0:00:00


In [ ]:
# 3. Unzip Code and Dataset Archives
import os

if os.path.exists('agrinexus_training_code.zip'):
    print("Unpacking code and checkpoints...")
    !unzip -q -o agrinexus_training_code.zip



Unpacking code and checkpoints...


In [ ]:
if os.path.exists('agrinexus_dataset.zip'):
    print("Unpacking dataset (this may take ~1-2 minutes)...")
    !unzip -q -o agrinexus_dataset.zip

print("Directory structure verified:")
!ls -la ml/disease/checkpoints/
!ls -la sample\ images/

Unpacking dataset (this may take ~1-2 minutes)...
Directory structure verified:
total 349060
drwxr-xr-x 2 root root      4096 Sep 11 10:05 .
drwxr-xr-x 3 root root      4096 Sep 11 10:05 ..
-rw-r--r-- 1 root root      7886 Sep 11 03:16 class_mapping.json
-rw-r--r-- 1 root root       408 Sep 11 03:11 epoch_log.csv
-rw-r--r-- 1 root root   6211071 Sep 10 23:54 gate1_model.pth
-rw-r--r-- 1 root root 351160565 Sep 11 03:11 last_epoch.pth
-rw-r--r-- 1 root root     35378 Sep  9 14:59 taxonomy.json
total 5284
drwxr-xr-x 2 root root    4096 Sep 11 10:13  .
drwxr-xr-x 1 root root    4096 Sep 11 10:13  ..
-rw-r--r-- 1 root root  744185 Sep 10 07:40  13re-garden-beech-05-kqcp-mediumSquareAt3X.jpg
-rw-r--r-- 1 root root  206506 Sep 10 07:40  dataset-card.jpg
-rw-r--r-- 1 root root   32593 Sep 10 01:44  images-2.jpeg
-rw-r--r-- 1 root root   31002 Sep 10 07:40  images-3.jpeg
-rw-r--r-- 1 root root    6704 Sep 10 01:44  images.jpeg
-rw-r--r-- 1 root root   66261 Sep 10 01:44  istockphoto-483451251-

In [ ]:
# 1. Set float16 for Colab's T4 GPU
!sed -i 's/"bfloat16"/"float16"/g' ml/disease/config.yaml

# 2. Run training on CUDA
!python ml/disease/train.py \
  --config ml/disease/config.yaml \
  --resume ml/disease/checkpoints/last_epoch.pth \
  --device cuda


2026-09-11 10:16:42,313 [INFO] numexpr.utils - NumExpr defaulting to 2 threads.
2026-09-11 10:16:43,521 [INFO] AgriNexusTrain - Compute device: cuda | AMP dtype: torch.float16 | GradScaler active: True
2026-09-11 10:16:43,528 [INFO] AgriNexusTrain - Initializing LeafDiseaseDataset on 'leaf_dataset/unified' with OOD from 'sample images'...
2026-09-11 10:16:43,530 [INFO] ml.disease.dataset - Persisted class mapping with 61 classes to ml/disease/checkpoints/class_mapping.json
2026-09-11 10:16:44,674 [WARNING] ml.disease.dataset - Class 'Tomato___Two_spotted_spider_mites' (idx 58) has only 2 sample(s) in split 'train' (< 10 threshold).
2026-09-11 10:16:44,674 [WARNING] ml.disease.dataset - Class 'background' (idx 60) has only 7 sample(s) in split 'train' (< 10 threshold).
2026-09-11 10:16:44,675 [INFO] ml.disease.dataset - LeafDiseaseDataset initialized: split='train', samples=40447, classes=61, broken_symlinks=0
2026-09-11 10:16:44,893 [WARNING] ml.disease.dataset - Class 'Bell_pepper___H

In [ ]:
# 5. Run Evaluation Harness on PlantDoc & OOD Distractor Set
!python ml/disease/evaluate.py \
  --checkpoint ml/disease/checkpoints/best_model.pth \
  --gate1-checkpoint ml/disease/checkpoints/gate1_model.pth \
  --config ml/disease/config.yaml \
  --device cuda

In [ ]:
# 6. Download the Trained Best Model and Logs
from google.colab import files

if os.path.exists('ml/disease/checkpoints/best_model.pth'):
    files.download('ml/disease/checkpoints/best_model.pth')
if os.path.exists('ml/disease/checkpoints/epoch_log.csv'):
    files.download('ml/disease/checkpoints/epoch_log.csv')